In [1]:
import numpy as np
import xarray as xr
import warnings
from pandas import date_range
from pathlib import Path

from grid_toolbox.basic_latlon import get_cells_area

import pycompo.core.composite as pccompo
import pycompo.core.coord as pccoord
import pycompo.core.ellipse as pcellipse
import pycompo.core.feature_cutout as pcfeatcut
import pycompo.core.filter as pcfilter
import pycompo.core.plot as pcplot
import pycompo.core.sst_features as pcsst
import pycompo.core.wind as pcwind
import pycompo.core.utils as pcutil

import matplotlib.pyplot as plt
import hfplot.figure.figure as hffig

warnings.filterwarnings(action='ignore')

/home/m/m300738/.conda/envs/TRR181L4/lib/python3.13/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


### Basic settings

In [ ]:
# read in configuration file
config_file = "/home/m/m300738/libs/pycompo/config/settings_ngc5004_opv4_w1.yaml"
config = pcutil.read_yaml_config(config_file)

start_time = config['data']['analysis_time'][0]
end_time = config['data']['analysis_time'][1]
analysis_times = [
    np.datetime64(t) for t in date_range(
        np.datetime64(start_time), np.datetime64(end_time), freq='MS',
        )
    ][2:4]
fdate_str = pcutil.create_ftime_str(analysis_times[0], analysis_times[1])
feature_var = config['data']['feature_var']
varlist = [feature_var] + config['data']['wind_vars']

### Data processing

In [ ]:
# read in, filter and subset data
infiles = []
for var in varlist:
    inpath = Path(config['data']['inpaths'][var])
    in_pattern = f"{config['exp']}_tropical_{var}_{fdate_str}.nc"
    infiles.extend(sorted([str(f) for f in inpath.rglob(in_pattern)]))
dset = xr.open_mfdataset(infiles, parallel=False).squeeze()
dset = pcutil.subsample_data(dset, start_time, end_time, config)
dset = dset.isel(time=slice(0, 2))
if 'height_2' in dset.coords: dset = dset.drop('height_2')

filter_vars = [feature_var] + config['data']['wind_vars']
dset_filter = \
    pcfilter.get_gaussian_filter_bg_ano(dset[filter_vars], **config['filter'])

dset = xr.merge([
    dset_filter[[f"{var}_bg" for var in config['data']['wind_vars']]],
    dset_filter[[f"{var}_ano" for var in filter_vars]],
    ])

dset['cell_area'] = get_cells_area(dset)
dset = dset.sel(lat=slice(*config['lat_range']), drop=True)
orig_coords = pccoord.get_coords_orig(dset.drop('time'))

# sample features and get feature properties
dset[f"{feature_var}_feature"], feature_props = pcsst.extract_sst_features(
    dset[f"{feature_var}_ano"], **config['feature'],
    )
feature_props, feature_data = pcfeatcut.get_featcen_data_cutouts(
    dset, feature_props, config['cutout']['search_RadRatio'],
    )
feature_props = pcwind.calc_feature_bg_wind(
    feature_props, feature_data, config['data']['wind_vars'],
    )
feature_props, feature_data = pcsst.add_more_feature_props(
    feature_props, feature_data, orig_coords,
    )

# create feature ellipse and filter with respect to aspect ratio
feature_ellipse = pcellipse.get_ellipse_params(feature_props, orig_coords)
feature_data, feature_props, feature_ellipse = pcsst.filter_asprat(
    feature_data, feature_props, feature_ellipse,
    max_asprat=config['feature']['max_ar'],
    )

# coordinate transformation
feature_data = pccoord.add_featcen_coords(
    orig_coords, feature_data, feature_props, feature_ellipse,
    )
feature_data = pcwind.add_rotate_winds(feature_data, feature_props)

feature_compo_data = pccompo.get_compo_coords_ds(feature_data, config)

feature_props = feature_props.where(
        feature_props['feature_id'].isin(feature_compo_data['feature_id']),
        drop=True,
    )

### Identify exemplary warm patches 

In [6]:
sample = feature_props.where(feature_props['asprat_cart'] < 1.5, drop=True)
sample = sample.where(sample['area_km2'] > 1500, drop=True)
sample = sample.where(sample['area_km2'] < 2500, drop=True)
sample

<xarray.Dataset> Size: 5kB
Dimensions:                (feature: 33, component: 2)
Coordinates:
  * component              (component) <U3 24B 'lat' 'lon'
    feature_id             (feature) int64 264B 357 659 831 ... 4126 4290 4314
Dimensions without coordinates: feature
Data variables: (12/14)
    radius_km              (feature) float64 264B 25.77 23.44 ... 26.22 24.44
    area_km2               (feature) float64 264B 2.086e+03 ... 1.877e+03
    time                   (feature) datetime64[ns] 264B 2020-05-01 ... 2020-...
    centroid_idx           (feature, component) float64 528B 56.0 ... 3.6e+03
    axis_major_length_idx  (feature) float64 264B 6.29 5.413 ... 6.775 6.161
    axis_minor_length_idx  (feature) float64 264B 4.542 4.261 ... 4.7 4.463
    ...                     ...
    bg_uas                 (feature) float64 264B -6.824 -3.224 ... -9.376 -6.61
    bg_vas                 (feature) float64 264B 2.796 -4.527 ... -2.513 0.471
    bg_sfcwind             (feature) float64 264B 7.375 5.558 ... 9.707 6.626
    bg_sfcwind_dir         (feature) float64 264B 112.3 35.46 ... 75.0 94.08
    centroid_sphere        (feature, component) float64 528B -10.02 ... 137.7
    asprat_cart            (feature) float64 264B 1.386 1.266 ... 1.459 1.401

### Plotting

In [ ]:
# basic settings
var = f"{feature_var}_ano"
dT_thresh = 0.15
feature_id = 357

level = 0.25

# ------------------------------------------------------------------------------
# get data
for data in feature_data:
    if data['feature_id'].values == feature_id:
        dset = data

winds = feature_props[['bg_uas', 'bg_vas']].where(
    feature_props['feature_id'] == feature_id, drop=True,
    )
fig, axs = hffig.init_subfig(
    style=('ams', 'two_column'), asprat=(6, 9.5), nrow=3, ncol=2,
    )

plt_ellipse = {}
for key, value in feature_ellipse.items():
    plt_ellipse[key] = value.where(
        value['feature_id'] == feature_id, drop=True,
        ).squeeze()

# ------------------------------------------------------------------------------
# Plot data in regular geophysical coordinates
axs[0, 0].pcolormesh(
    dset['lon'], dset['lat'], dset[var], cmap='RdBu_r', vmin=-level, vmax=level,
    )
axs[0, 0].contour(
    dset['lon'], dset['lat'], dset[var], levels=[dT_thresh], colors='beige',
    )
axs[0, 0].scatter(
    plt_ellipse['sphere']['centroid'].sel(component='lon'),
    plt_ellipse['sphere']['centroid'].sel(component='lat'),
    color='beige',
    )
axs[0, 0].set_title('Regular geophysical\ncoordinate')
axs[0, 0].set_xlabel('Longitude / $^{\circ}\,$E')
axs[0, 0].set_ylabel('Latitude / $^{\circ}\,$N')


# Plot data in feature-centric geophysical coordinates
axs[0, 1].pcolormesh(
    dset['featcen_lon'], dset['featcen_lat'], dset[var],
    cmap='RdBu_r', vmin=-level, vmax=level,
    )
axs[0, 1].contour(
    dset['featcen_lon'], dset['featcen_lat'], dset[var],
    levels=[dT_thresh], colors='beige',
    )
axs[0, 1].scatter(
    plt_ellipse['featcen_sphere']['centroid'].sel(component='lon'),
    plt_ellipse['featcen_sphere']['centroid'].sel(component='lat'),
    color='beige',
    )
axs[0, 1].set_title('Feature-centric geophysical\ncoordinate')
axs[0, 1].set_xlabel('Feature-centric longitude / $^{\circ}\,$E')
axs[0, 1].set_ylabel('Feature-centric latitude / $^{\circ}\,$N')


# Plot data in feature-centric Cartesian coordinates
axs[1, 0].pcolormesh(
    dset['featcen_x'], dset['featcen_y'], dset[var], cmap='RdBu_r',
    vmin=-level, vmax=level,
    )
axs[1, 0].contour(
    dset['featcen_x'], dset['featcen_y'], dset[var],
    levels=[dT_thresh], colors='beige',
    )
axs[1, 0].scatter(
    plt_ellipse['featcen_cart']['centroid'].sel(component='x'),
    plt_ellipse['featcen_cart']['centroid'].sel(component='y'),
    color='beige',
    )
axs[1, 0].set_title('Feature-centric Cartesian\ncoordinate')
axs[1, 0].set_xlabel('Feature-centric distance / km')
axs[1, 0].set_ylabel('Feature-centric distance / km')


# Plot data in rotated feature-centric Cartesian coordinates
axs[1, 1].pcolormesh(
    dset['rota_featcen_x'], dset['rota_featcen_y'], dset[var],
    cmap='RdBu_r', vmin=-level, vmax=level,
    )
axs[1, 1].contour(
    dset['rota_featcen_x'], dset['rota_featcen_y'], dset[var],
    levels=[dT_thresh], colors='beige',
    )
axs[1, 1].scatter(
    plt_ellipse['rota_featcen_cart']['centroid'].sel(component='x'),
    plt_ellipse['rota_featcen_cart']['centroid'].sel(component='y'),
    color='beige',
    )
axs[1, 1].set_title('Rotated feature-centric\nCartesian coordinate')
axs[1, 1].set_xlabel('Feature-centric distance / km')
axs[1, 1].set_ylabel('Feature-centric distance / km')


# Plot data in normalized rotated feature-centric Cartesian coordinates
axs[2, 0].pcolormesh(
    dset['En_rota_featcen_x'], dset['En_rota_featcen_y'], dset[var],
    cmap='RdBu_r', vmin=-level, vmax=level,
    )
axs[2, 0].contour(
    dset['En_rota_featcen_x'], dset['En_rota_featcen_y'], dset[var],
    levels=[dT_thresh], colors='beige',
    )
axs[2, 0].scatter([0], [0], color='beige')
axs[2, 0].set_title('Normalized rotated feature-\ncentric Cartesian coordinate')
axs[2, 0].set_xlabel('Fractional distance')
axs[2, 0].set_ylabel('Fractional distance')


# Plot data in normalized rotated feature-centric Cartesian coordinates
pl1 = axs[2, 1].pcolormesh(
    dset['En_rota2_featcen_x'], dset['En_rota2_featcen_y'], dset[var],
    cmap='RdBu_r', vmin=-level, vmax=level,
    )
axs[2, 1].contour(
    dset['En_rota2_featcen_x'], dset['En_rota2_featcen_y'], dset[var],
    levels=[dT_thresh], colors='beige',
    )
axs[2, 1].scatter([0], [0], color='beige')
axs[2, 1].set_title('Wind-aligned feature-centric\nCartesian coordinate')
axs[2, 1].set_xlabel('Downwind fractional distance')
axs[2, 1].set_ylabel('Downwind fractional distance')


# ------------------------------------------------------------------------------
# Plot grid lines
pcplot.plot_feature_grid(
    axs[0, 1], dset['featcen_lon'], dset['featcen_lat'],
    )
pcplot.plot_feature_grid(
    axs[1, 0], dset['featcen_x'], dset['featcen_y'],
    )
pcplot.plot_feature_grid(
    axs[1, 1], dset['rota_featcen_x'], dset['rota_featcen_y'],
    )
pcplot.plot_feature_grid(
    axs[2, 0], dset['En_rota_featcen_x'], dset['En_rota_featcen_y'],
    )
pcplot.plot_feature_grid(
    axs[2, 1], dset['En_rota2_featcen_x'], dset['En_rota2_featcen_y'],
    )

# Plot feature ellipses/circles
pcplot._plot_feature_ellipse(
    axs[0, 1], plt_ellipse['featcen_sphere'],
    )
pcplot._plot_feature_ellipse(
    axs[1, 0], plt_ellipse['featcen_cart'],
    )
pcplot._plot_feature_ellipse(
    axs[1, 1], plt_ellipse['rota_featcen_cart'],
    )
pcplot.plot_feature_circle(axs[2, 0], (0, 0), 1, color='k', lw=1)
pcplot.plot_feature_circle(axs[2, 1], (0, 0), 1, color='k', lw=1)


# ------------------------------------------------------------------------------
# Plot wind features
color='k'
hw=5
axs[0, 0].quiver(
    plt_ellipse['sphere']['centroid'].sel(component='lon'),
    plt_ellipse['sphere']['centroid'].sel(component='lat'),
    winds['bg_uas'], winds['bg_vas'],
    scale=50, zorder=2, color=color, headwidth=hw,
    )
axs[0, 1].quiver(
    0, 0, winds['bg_uas'], winds['bg_vas'],
    scale=50, zorder=2, color=color, headwidth=hw,
    )
axs[1, 0].quiver(
    0, 0, winds['bg_uas'], winds['bg_vas'],
    scale=50, zorder=2, color=color, headwidth=hw,
    )

uas_rota, vas_rota = pccoord._calc_rota_featcen_cart_coords(
    winds['bg_uas'], winds['bg_vas'],
    plt_ellipse['featcen_cart']['polar_angle_rad'],
    )
axs[1, 1].quiver(
    0, 0, uas_rota, vas_rota, scale=50, zorder=2, color=color, headwidth=hw,
    )
axs[2, 0].quiver(
    0, 0, uas_rota, vas_rota, scale=50, zorder=2, color=color, headwidth=hw,
    )

uas_rota2, vas_rota2 = pccoord._calc_rota_featcen_cart_coords(
    winds['bg_uas'], winds['bg_vas'],
    np.arctan2(winds['bg_vas'], winds['bg_uas']),
    )
axs[2, 1].quiver(
    0, 0, uas_rota2, vas_rota2, scale=50, zorder=2, color=color, headwidth=hw
    )


# ------------------------------------------------------------------------------
# set ticks
ticks = np.arange(-1.2, 1.21, 0.6)
axs[0, 1].set_xticks(ticks)
axs[0, 1].set_yticks(ticks)

ticks = np.arange(-150, 151, 75)
axs[1, 0].set_xticks(ticks)
axs[1, 0].set_yticks(ticks)

ticks = np.arange(-200, 201, 100)
axs[1, 1].set_xticks(ticks)
axs[1, 1].set_yticks(ticks)

for i in [0, 1]:
    axs[2, i].set_xlim([-4, 4])
    axs[2, i].set_ylim([-4, 4])
    axs[2, i].set_yticks(np.arange(-4, 4.2, 2))

# set spines
for ax in axs.ravel():
    for spine in ax.spines.values():
        spine.set_visible(False)

# add colorbar
cbar_ax = fig.add_axes([0.26, 0.04, 0.6, 0.015])
cbar = fig.colorbar(
    pl1, cax=cbar_ax, orientation='horizontal',
    label="$SST'$ / K", extend='both', ticks=np.arange(-0.2, 0.25, 0.1)
)
cbar.ax.axvline(0.15, color='beige', linewidth=1)

# make plot nice
plt.tight_layout()
fig.subplots_adjust(bottom=0.12)
for i in range(0, len(axs.ravel())): axs.ravel()[i].set_aspect('equal')

# saving plot
outpath = Path(f'/home/m/m300738/project_TRR181L4/plots/paper/')
outpath.mkdir(parents=True, exist_ok=True)
outfile = Path('figureA2.png')
plt.savefig(str(outpath/outfile), dpi=600)
plt.show()